# Let's learn matplotlib with Nvidia's timeline

A recipe: 27 years of Nvidia's share price, the chips that built it, and the scandal of November 2026.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/shol-recipes/blob/main/01-nvidia-matplotlib/nvidia_timeline.ipynb)

## 0. Install
Only needed the first time (or in Colab).

In [ ]:
python3 -m venv .venv
source .venv/bin/activate
pip install ipykernel yfinance matplotlib pandas

In [ ]:
!pip install -q yfinance matplotlib pandas

## 1. Ingredients
Daily closing price since the IPO in January 1999.

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

df = yf.download("NVDA", start="1999-01-22", end="2026-11-13", auto_adjust=True)
close = df["Close"].squeeze()
close.tail()

## 2. The menu
Two dictionaries that map dates to labels.

In [ ]:
products = {
    "1999-08-31": "GeForce 256",
    "2006-11-08": "CUDA",
    "2012-09-30": "AlexNet trains on GeForce",
    "2016-04-05": "DGX-1",
    "2020-05-14": "A100",
    "2022-03-22": "H100",
    "2024-03-18": "Blackwell",
}
scandal = {
    "2026-10-29": "NYT report",
    "2026-11-08": "Affair admitted",
    "2026-11-11": "Resignation accepted",
}

## 3. Plate 1: the whole story
Two charts side by side. Log scale so 1999 and 2026 both fit. Change the pink to any colour you like.

In [ ]:
plt.style.use("dark_background")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6),
                               gridspec_kw={"width_ratios": [3, 1]})

ax1.plot(close.index, close, color="#ff69b4", linewidth=1.4)
ax1.set_yscale("log")
for date, label in products.items():
    d = pd.Timestamp(date)
    y = close.asof(d)
    ax1.scatter(d, y, color="white", s=20, zorder=3)
    ax1.annotate(label, (d, y), xytext=(0, 25), textcoords="offset points",
                 ha="center", fontsize=8, color="white",
                 arrowprops=dict(arrowstyle="-", color="grey"))
ax1.axvspan(pd.Timestamp("2026-10-01"), pd.Timestamp("2026-11-12"),
            color="red", alpha=0.3)
ax1.set_title("1999 to now")
ax1.set_ylabel("Share price, USD (split-adjusted, log scale)")
ax1.xaxis.set_major_locator(mdates.YearLocator(3))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.close(fig)
fig

## 4. Plate 2: zoom on the scandal
Two weeks are invisible on a 27-year chart, so we zoom in.

In [ ]:
recent = close["2026-10-01":]
ax2.plot(recent.index, recent, color="#ff69b4", linewidth=2)
for date, label in scandal.items():
    d = pd.Timestamp(date)
    ax2.axvline(d, color="#ff9999", linestyle="--", linewidth=1)
    ax2.text(d, ax2.get_ylim()[1], " " + label, rotation=90, va="top",
             fontsize=8, color="#ff9999")
ax2.set_title("October to November 2026")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax2.tick_params(axis="x", rotation=45)
fig

## 5. Garnish and serve

In [ ]:
for ax in (ax1, ax2):
    ax.grid(alpha=0.15)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
fig.suptitle("NVDA: from GeForce to the boardroom", fontsize=15)
fig.tight_layout()
fig.savefig("nvidia_timeline.png", dpi=200)
fig

## Try it yourself
- Swap `"NVDA"` for `"AMD"` and plot both on the same axes
- Add a third plate for trading volume with `ax.bar()`
- Mark the 17 November earnings with another `axvline`